In [ ]:
import pandas as pd


full_data = pd.read_excel(r"D:\Downloads\DSTC vòng 3\raw data\UPCOMINDEX.xlsx")
full_data.info()

In [ ]:
from FiinQuantX import FiinSession
import pandas as pd
username = __import__('os').environ['FIINQUANT_USERNAME']
password = __import__('os').environ['FIINQUANT_PASSWORD']

client = FiinSession(username=username, password=password).login()
fi = client.FiinIndicator()

In [ ]:
full_data = full_data.drop(["bu", "sd", "fn", "fb", "fs"], axis = 1)
full_data.info()

In [ ]:
counts = full_data.groupby("ticker").size()
counts.describe()

In [ ]:
list_keep = counts[counts == 919].index
list_keep.size

In [ ]:
df = full_data[full_data['ticker'].isin(list_keep)].reset_index()
df = df.drop("Unnamed: 0", axis = 1)
df = df.drop("index", axis = 1)
df.info()

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
zero_counts_per_ticker = df["volume"].eq(0).groupby(df["ticker"]).sum()

# 2) Lấy 16 ticker có số lần volume=0 ít nhất
min_nonzero = zero_counts_per_ticker[zero_counts_per_ticker > 0]
tickers_min_16 = min_nonzero.nsmallest(16).index

df_zero_min16 = df[(df["ticker"].isin(tickers_min_16)) & (df["volume"] == 0)]

# 4) Lấy danh sách timestamp
timestamps_min16 = df_zero_min16["timestamp"].unique()


print("Timestamp mà 16 ticker ít volume=0 nhất có volume=0:")
print(timestamps_min16)

In [ ]:
df_filtered = df[~df["timestamp"].isin(timestamps_min16)]
df_filtered["timestamp"].nunique()

In [ ]:
tickers_with_zero = df_filtered.loc[df_filtered['volume'] == 0, 'ticker'].unique()
print(tickers_with_zero.size)

In [ ]:
df_clean = df_filtered[~df_filtered['ticker'].isin(tickers_with_zero)].copy()
df_clean.info()

In [ ]:
print(df_clean['ticker'].unique())

In [ ]:
df_clean = df_clean.reset_index()
df_clean = df_clean.drop("index", axis =1)


In [ ]:
df_clean.info()

In [ ]:
df_clean.tail(20)

In [ ]:
import numpy as np

tickers = df_clean['ticker'].unique()
all_dfs = []

    
for t in tickers:
    df_t = df_clean[df_clean['ticker'] == t].copy()

    df_t['ema_50'] = fi.ema(df_t['close'], window=50)
    df_t['ema_200'] = fi.ema(df_t['close'], window=200)
    df_t['macd'] = fi.macd(df_t['close'], window_fast=12, window_slow=26)
    df_t['macd_signal'] = fi.macd_signal(df_t['close'], window_fast=12, window_slow=26, window_sign=9)
    df_t['macd_diff'] = fi.macd_diff(df_t['close'], window_fast=12, window_slow=26, window_sign=9)
    df_t['rsi'] = fi.rsi(df_t['close'], window=14) 
    df_t['bollinger_hband'] = fi.bollinger_hband(df_t['close'], window=20, window_dev=2)
    df_t['bollinger_lband'] = fi.bollinger_lband(df_t['close'], window=20, window_dev=2)
    df_t['mfi'] = fi.mfi(df_t['high'], df_t['low'], df_t['close'], df_t['volume'], window=14) 

    df_t['ema_200'] = np.where(df_t['ema_200'].isna(), df_t['ema_50'], df_t['ema_200'])

    all_dfs.append(df_t)



df_indicators = pd.concat(all_dfs, ignore_index=True)

print(df_indicators.info())

In [ ]:
df_indicators.head(20)

In [ ]:
df_cleaned = df_indicators.dropna(how = 'any').copy().reset_index()
df_cleaned.info()

In [ ]:
df_cleaned = df_cleaned.drop("index", axis =1)
df_cleaned.head()

In [ ]:
df_cleaned['golden_cross'] = df_cleaned.groupby('ticker').apply(
    lambda x: ((x['ema_50'] > x['ema_200']) & (x['ema_50'].shift(1) <= x['ema_200'].shift(1))).astype(int)
).reset_index(level=0, drop=True)

df_cleaned['death_cross'] = df_cleaned.groupby('ticker').apply(
    lambda x: ((x['ema_50'] < x['ema_200']) & (x['ema_50'].shift(1) >= x['ema_200'].shift(1))).astype(int)
).reset_index(level=0, drop=True)

df_cleaned['return'] = df_cleaned.groupby('ticker')['close'].pct_change()

df_cleaned['macd_cross'] = df_cleaned.groupby('ticker').apply(
    lambda x: ((x['macd'] > x['macd_signal']) & (x['macd'].shift(1) <= x['macd_signal'].shift(1))).astype(int) -
              ((x['macd'] < x['macd_signal']) & (x['macd'].shift(1) >= x['macd_signal'].shift(1))).astype(int)
).reset_index(level=0, drop=True)

df_cleaned['bollinger_pct'] = (df_cleaned['close'] - df_cleaned['bollinger_lband']) / (df_cleaned['bollinger_hband'] - df_cleaned['bollinger_lband'])

df_cleaned['bollinger_bw'] = (df_cleaned['bollinger_hband'] - df_cleaned['bollinger_lband']) / ((df_cleaned['bollinger_hband'] + df_cleaned['bollinger_lband']) / 2)


df_cleaned.head()

In [ ]:
df_cleaned.dropna(inplace=True)
df_cleaned = df_cleaned.reset_index()
df_cleaned.info()

In [ ]:
df_cleaned.columns

In [ ]:
df_cleaned = df_cleaned.drop("index", axis =1)

In [ ]:
df_cleaned.head()

In [ ]:
df_cleaned.to_excel(r"cleaned data\UPCOM_cleaned.xlsx")